In [1]:
import ase.io
import io
from rdkit.Chem.rdMolAlign import GetBestRMS
import rdkit.Chem

In [2]:
all_atoms = ase.io.read('/tmp/sample_selected_TS-20250703-SCAN-8-w_wo_selfloops-lr2_5_lr5_0e-4-ValFix3/rcmconly_passerini-TS13605.xyz', index=':')

In [3]:
#all_atoms[-1] = ase.io.read('/tmp/test_ref.xyz')

In [4]:
#all_atoms[1] = ase.io.read('/tmp/test2.xyz')

In [5]:
mols = []
for atoms in all_atoms:
    with io.StringIO() as file:
        atoms.write(file, format='xyz', comment=' '.join(atoms.info))
        mol = rdkit.Chem.MolFromXYZBlock(file.getvalue())
        mols.append(mol)

In [6]:
GetBestRMS(mols[-1], mols[1])

1.879275964556519

In [7]:
aligned_atoms = []
for mol in [mols[1], mols[-1]]:
    with io.StringIO(rdkit.Chem.MolToXYZBlock(mol)) as file:
        atoms = ase.io.read(file, format='xyz')
        aligned_atoms.append(atoms)

In [8]:
from ase.visualize import view
atoms1 = aligned_atoms[0]
atoms2 = aligned_atoms[1]
#atoms2.positions += atoms1.positions[5] - atoms2.positions[5]
#atoms2.rotate(-atoms2.positions[5]+atoms2.positions[0], -atoms1.positions[5]+atoms1.positions[6], center=atoms2.positions[5])
#atoms2.rotate(-atoms2.positions[5]+atoms2.positions[0], 90, center=atoms2.positions[5])
v = view(atoms1+atoms2, viewer='ngl')
v.view.remove_spacefill()
v.view.add_ball_and_stick(selection=range(len(atoms1)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent')
v.view.add_ball_and_stick(selection=range(len(atoms1),len(atoms1)+len(atoms2)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
v

In [9]:
import rmsd
import numpy as np
from ase.build import minimize_rotation_and_translation

In [10]:
order = rmsd.reorder_hungarian(np.array(atoms1.get_chemical_symbols()), np.array(atoms2.get_chemical_symbols()), atoms1.positions, atoms2.positions)

In [11]:
atoms2 = atoms2[order]
minimize_rotation_and_translation(atoms1, atoms2)

In [12]:
from ase.visualize import view
v = view(atoms1+atoms2, viewer='ngl')
v.view.remove_spacefill()
v.view.add_ball_and_stick(selection=range(len(atoms1)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent')
v.view.add_ball_and_stick(selection=range(len(atoms1),len(atoms1)+len(atoms2)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
v

In [13]:
rmsd = np.sum((atoms1.positions - atoms2.positions)**2, axis=-1).mean()**0.5
rmsd

1.518493646064896

In [14]:
from pymatgen.core import Molecule
mol1 = Molecule.from_ase_atoms(atoms1)
mol2 = Molecule.from_ase_atoms(atoms2)

In [15]:
from pymatgen.analysis import molecule_matcher

In [16]:
matcher = molecule_matcher.HungarianOrderMatcher(mol1)
matcher.match(mol2)

(array([15, 17, 18, 14,  6,  5,  4,  3,  8,  7,  0, 11, 12,  2,  9, 10,  1,
        16, 13, 19]),
 array([[-0.40573564,  0.41969186, -0.81193432],
        [-0.91040933, -0.10701692,  0.39962762],
        [ 0.08082975,  0.90133574,  0.42551196]]),
 array([-0.00450836, -0.05879369,  0.20579719]),
 0.9947714650512426)

In [17]:
from ase.visualize import view
v = view(mol1.to_ase_atoms()+mol2.to_ase_atoms(), viewer='ngl')
v.view.remove_spacefill()
v.view.add_ball_and_stick(selection=range(len(atoms1)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent')
v.view.add_ball_and_stick(selection=range(len(atoms1),len(atoms1)+len(atoms2)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
v

In [36]:
matcher = molecule_matcher.GeneticOrderMatcher(mol1, threshold=0.5)
mol2, rmsd = min(matcher.fit(mol2), key=lambda x:x[-1])
rmsd

0.34385701676723046

In [30]:
from ase.visualize import view
v = view(mol1.to_ase_atoms()+mol2.to_ase_atoms(), viewer='ngl')
v.view.remove_spacefill()
v.view.add_ball_and_stick(selection=range(len(atoms1)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent')
v.view.add_ball_and_stick(selection=range(len(atoms1),len(atoms1)+len(atoms2)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
v

In [32]:
all_atoms[-1].info

{'Generated/inpainted': True,
 'transition': True,
 'state.': True,
 'RMSD:': True,
 '0.343857': True,
 'Å.': True,
 'By': True,
 'model': True,
 '/misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/8w-lr5e-4-ValFix3-SCAN-leftnetefadd8bd15e7/ddpm-epoch': '1697-val-totloss=655.96.ckpt.'}